In [ ]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestNeighbors

import warnings
warnings.filterwarnings('ignore')

In [ ]:
data_books = pd.read_csv('Books.csv')
data_ratings = pd.read_csv('Ratings.csv')
data_user = pd.read_csv('Users.csv', nrows=160486)

# Nova seção

In [ ]:
data = data_books.merge(data_ratings, how='inner', on='ISBN')

In [ ]:
data = data.merge(data_user, how='inner', on='User-ID')

In [ ]:
data['Year-Of-Publication'] = pd.to_numeric(data['Year-Of-Publication'], errors='coerce')
data['Age'] = pd.to_numeric(data['Age'], errors='coerce')

In [ ]:
data[data['Year-Of-Publication'].apply(lambda x: isinstance(x, str))] = ''

In [ ]:
data['Location'] = data['Location'].str.split(',').str[-1]

In [ ]:
data = data.loc[data['Book-Rating'] > 0]
data = data.loc[data['Year-Of-Publication'] >= 1965]
data = data.loc[data['Year-Of-Publication'] <= 2023]
data = data.loc[data['Age'] <= 90]
data = data.loc[data['Age'] >= 7]

In [ ]:
data['Year-Of-Publication'] = data['Year-Of-Publication'].astype(int)

In [ ]:
matriz = data.pivot_table(values='Book-Rating', index='Book-Title', columns='User-ID')
matriz.fillna(0, inplace=True)

from scipy.sparse import csc_matrix
matriz_esparsa = csc_matrix(matriz)

In [ ]:
model = NearestNeighbors(n_neighbors=5,
                         algorithm='auto',
                         metric="minkowski")
model.fit(matriz_esparsa)

NearestNeighbors()

In [ ]:
for posicao, titulo in enumerate(matriz.index):
    if 'Harry Potter' in str(titulo):
        print(posicao, titulo)

8568 Harrius Potter et Philosophi Lapis (Harry Potter and the Philosopher's Stone, Latin edition)
8569 Harry Potter Collector's Value Guide
8570 Harry Potter E il Prigioniero D'Azkaban
8571 Harry Potter E la Camera Dei Segreti
8572 Harry Potter Et LA Coupe De Feu
8573 Harry Potter Schoolbooks: Quidditch Through the Ages and Fantastic Beasts and Where to Find Them
8574 Harry Potter Stationery Kit
8575 Harry Potter Und Der Feuerkelch
8576 Harry Potter Y El Prisionero De Azkaban
8577 Harry Potter and the Chamber of Secrets
8578 Harry Potter and the Chamber of Secrets (Book 2 Audio CD)
8579 Harry Potter and the Chamber of Secrets (Book 2)
8580 Harry Potter and the Goblet of Fire (Book 4 Audio CD)
8581 Harry Potter and the Goblet of Fire (Book 4)
8582 Harry Potter and the Order of the Phoenix (Book 5)
8583 Harry Potter and the Prisoner of Azkaban (Book 3 Audio CD)
8584 Harry Potter and the Prisoner of Azkaban (Book 3)
8585 Harry Potter and the Prisoner of Azkaban (Book 3, Audio)
8586 Harry 

In [ ]:
livro = matriz_esparsa[8596, :].reshape(1, -1)

In [ ]:
distancia, recomendacao = model.kneighbors(livro)

In [ ]:
distancia

array([[ 0.        , 18.92088793, 20.32240143, 20.63976744, 20.63976744]])

In [ ]:
recomendacao

array([[ 8596,  8595,  8597, 12081,  6321]])

In [ ]:
for livro in recomendacao:
    print(matriz.index[livro])

Index(['Harry Potter y la cÃ¡mara secreta',
       'Harry Potter y el prisionero de Azkaban',
       'Harry Potter y la piedra filosofal', 'Mafalda  1',
       'El Libro Negro de Las Grandes Marcas'],
      dtype='object', name='Book-Title')
